# Food vs. Not-Food Detector V2 - Low-Disk Colab Training

Accuracy-focused binary classifier for Nutrify using a compact public ImageNet-derived dataset. It contains 8,000 images: 4,000 food and 4,000 not-food examples from the same source.

**Resource target:** about 1 GB working disk, a Google Colab T4, and approximately 30-90 minutes depending on fine-tuning and early stopping.

**Deployment contract**

- Input: float32 RGB `[batch, 224, 224, 3]` in `[0, 1]`.
- Output: one float32 sigmoid score where `0 = not food` and `1 = food`.
- Output model: `food_or_not_food_detector_v2.keras`.

This compact dataset is narrower than Food-101 plus COCO. Treat its test metrics as a controlled benchmark, then test with real phone-camera images before production deployment.

In [ ]:
# Remove the large local TFDS/COCO cache created by earlier notebook versions.
import shutil
from pathlib import Path

old_pipeline_dir = Path("/content/nutrify_food_detector_v2")
if old_pipeline_dir.exists():
    shutil.rmtree(old_pipeline_dir)
    print("Removed old high-disk dataset cache:", old_pipeline_dir)
else:
    print("No old high-disk cache found")


In [ ]:
%pip install -q "imagehash>=4.3.1" "pybktree>=1.1" "pyarrow>=14" "seaborn>=0.13"


In [ ]:
import hashlib
import io
import json
import os
import random
import shutil
import time
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pybktree
import requests
import seaborn as sns
import tensorflow as tf
from PIL import Image
from google.colab import drive
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve,
)
from tqdm.auto import tqdm

SEED = 601
IMG_SIZE = 224
BATCH_SIZE = 32
BACKBONE_NAME = "EfficientNetV2S"
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 24
HARD_EXAMPLE_EPOCHS = 6
MAX_HARD_PER_CLASS = 200
MIN_FOOD_RECALL = 0.97
TARGET_BALANCED_ACCURACY = 0.95
TARGET_SPECIFICITY = 0.93
CPU_SPEED_FLOOR_MS = 300.0
PHASH_DISTANCE = 4
TRAINING_RECIPE_VERSION = 1
RESUME_FROM_DRIVE = True

DATASET_URL = (
    "https://huggingface.co/datasets/avnishs17/food_not_food/"
    "resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"
)
DATASET_REVISION = "29c88c23e7ae88ce1ef3ce6dec3c19617fc72c58"
EXPECTED_ROWS = 8000

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
tf.keras.mixed_precision.set_global_policy("mixed_float16")

gpus = tf.config.list_physical_devices("GPU")
free_disk_gb = shutil.disk_usage("/content").free / (1024 ** 3)
print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus)
print(f"Free local disk: {free_disk_gb:.1f} GB")
if not gpus:
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU.")
if free_disk_gb < 2:
    raise RuntimeError("At least 2 GB of free local disk is required.")


In [ ]:
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/NutrifyFoodDetectorV2LowDisk")
ARTIFACT_DIR = DRIVE_ROOT / "artifacts"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
MANIFEST_DIR = DRIVE_ROOT / "manifests"
LOCAL_ROOT = Path("/content/nutrify_food_detector_v2_low_disk")
IMAGE_DIR = LOCAL_ROOT / "images"
PARQUET_PATH = LOCAL_ROOT / "food_not_food.parquet"
RAW_MANIFEST_PATH = MANIFEST_DIR / "raw_image_manifest.csv"
SPLIT_MANIFEST_PATH = MANIFEST_DIR / "split_manifest.csv"

for directory in (ARTIFACT_DIR, CHECKPOINT_DIR, MANIFEST_DIR, IMAGE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Drive artifacts:", ARTIFACT_DIR)
print("Local images:", IMAGE_DIR)


## Download and materialize the compact dataset

The notebook streams one 280 MB Parquet file, writes the original encoded images locally, computes exact and perceptual hashes, then deletes the Parquet file. Peak working disk remains below about 1 GB.

The source labels are `0 = food`, `1 = not_food`; this cell deliberately converts them to Nutrify's contract: `1 = food`, `0 = not_food`.

In [ ]:
def download_file(url, destination):
    temporary = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with temporary.open("wb") as output, tqdm(
            total=total, unit="B", unit_scale=True, desc=destination.name
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))
    temporary.replace(destination)


def hash_image(encoded_bytes):
    sha256 = hashlib.sha256(encoded_bytes).hexdigest()
    with Image.open(io.BytesIO(encoded_bytes)) as image:
        image = image.convert("RGB")
        phash = str(imagehash.phash(image))
    return sha256, phash


local_images_ready = (
    RAW_MANIFEST_PATH.exists()
    and len(list(IMAGE_DIR.glob("*.img"))) == EXPECTED_ROWS
)

if local_images_ready:
    raw_manifest = pd.read_csv(RAW_MANIFEST_PATH)
    local_images_ready = raw_manifest["path"].map(lambda value: Path(value).exists()).all()

if not local_images_ready:
    shutil.rmtree(IMAGE_DIR, ignore_errors=True)
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    if not PARQUET_PATH.exists():
        download_file(DATASET_URL, PARQUET_PATH)

    parquet = pq.ParquetFile(PARQUET_PATH)
    rows = []
    index = 0
    for batch in tqdm(parquet.iter_batches(batch_size=128, columns=["image", "label"]), desc="Materializing"):
        data = batch.to_pydict()
        for image_record, source_label in zip(data["image"], data["label"]):
            encoded = image_record["bytes"]
            if not encoded:
                raise RuntimeError(f"Missing image bytes at row {index}")
            path = IMAGE_DIR / f"{index:06d}.img"
            path.write_bytes(encoded)
            try:
                sha256, phash = hash_image(encoded)
            except Exception as error:
                path.unlink(missing_ok=True)
                print(f"Skipping corrupt row {index}: {error}")
                index += 1
                continue
            app_label = 1 if int(source_label) == 0 else 0
            rows.append({
                "record_id": f"public:{index}",
                "path": str(path),
                "label": app_label,
                "source_label": int(source_label),
                "sha256": sha256,
                "phash": phash,
            })
            index += 1
    raw_manifest = pd.DataFrame(rows)
    raw_manifest.to_csv(RAW_MANIFEST_PATH, index=False)
    PARQUET_PATH.unlink(missing_ok=True)
else:
    print("Reusing materialized images and Drive manifest")

print(raw_manifest["label"].value_counts().rename(index={0: "not_food", 1: "food"}))
assert set(raw_manifest["label"].unique()) == {0, 1}
assert len(raw_manifest) >= 7900, "Too many source images were missing or corrupt"


## Duplicate isolation and deterministic splits

Exact duplicates and perceptual near-duplicates are grouped before splitting. Mixed-label duplicate groups are removed. One canonical image remains per group. Each class is independently divided into 60% train, 10% validation, 10% calibration, 10% untouched test, and 10% hard-example mining.

In [ ]:
manifest = raw_manifest.copy().reset_index(drop=True)
parent = np.arange(len(manifest), dtype=np.int64)
rank = np.zeros(len(manifest), dtype=np.int8)

def find(item):
    while parent[item] != item:
        parent[item] = parent[parent[item]]
        item = parent[item]
    return int(item)

def union(left, right):
    left_root, right_root = find(left), find(right)
    if left_root == right_root:
        return
    if rank[left_root] < rank[right_root]:
        left_root, right_root = right_root, left_root
    parent[right_root] = left_root
    if rank[left_root] == rank[right_root]:
        rank[left_root] += 1

exact_seen = {}
for position, sha256 in enumerate(manifest["sha256"]):
    previous = exact_seen.setdefault(sha256, position)
    if previous != position:
        union(position, previous)

def phash_distance(left, right):
    return (left[0] ^ right[0]).bit_count()

tree = pybktree.BKTree(phash_distance)
for position, phash in enumerate(tqdm(manifest["phash"], desc="Near duplicates")):
    item = (int(phash, 16), position)
    for _, (_, previous) in tree.find(item, PHASH_DISTANCE):
        union(position, previous)
    tree.add(item)

manifest["group_id"] = [find(position) for position in range(len(manifest))]
mixed_groups = set(
    manifest.groupby("group_id")["label"].nunique().loc[lambda values: values > 1].index
)
if mixed_groups:
    manifest[manifest["group_id"].isin(mixed_groups)].to_csv(
        MANIFEST_DIR / "ambiguous_duplicate_groups.csv", index=False
    )
    manifest = manifest[~manifest["group_id"].isin(mixed_groups)].copy()

manifest["stable_order"] = manifest["record_id"].map(
    lambda value: hashlib.sha256(f"{SEED}:{value}".encode()).hexdigest()
)
manifest = (
    manifest.sort_values(["group_id", "stable_order"])
    .drop_duplicates("group_id", keep="first")
    .copy()
)

partition_by_id = {}
for label, label_rows in manifest.groupby("label"):
    ordered = label_rows.sort_values("stable_order")
    count = len(ordered)
    boundaries = [int(count * value) for value in (0.60, 0.70, 0.80, 0.90)]
    names = ("train", "validation", "calibration", "test", "mining")
    sections = np.split(ordered.index.to_numpy(), boundaries)
    for name, indices in zip(names, sections):
        partition_by_id.update({int(index): name for index in indices})
manifest["partition"] = manifest.index.map(partition_by_id)
manifest = manifest.drop(columns="stable_order")
manifest.to_csv(SPLIT_MANIFEST_PATH, index=False)

split_summary = pd.crosstab(manifest["partition"], manifest["label"])
display(split_summary.rename(columns={0: "not_food", 1: "food"}))
assert not manifest.groupby("group_id")["partition"].nunique().gt(1).any()
assert (split_summary > 0).all().all()

manifest_fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(
        manifest[["record_id", "sha256", "label", "partition"]], index=False
    ).values.tobytes()
).hexdigest()
run_config = {
    "dataset_revision": DATASET_REVISION,
    "manifest_fingerprint": manifest_fingerprint,
    "backbone": BACKBONE_NAME,
    "image_size": IMG_SIZE,
    "seed": SEED,
    "training_recipe_version": TRAINING_RECIPE_VERSION,
}
RUN_SIGNATURE = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()[:16]
RUN_CHECKPOINT_DIR = CHECKPOINT_DIR / RUN_SIGNATURE
RUN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Run signature:", RUN_SIGNATURE)


## Input pipelines

Pillow nearest-neighbor preprocessing intentionally matches `main.py`: RGB decode, warped `224 x 224` resize, divide by 255, and batch. Training samples both labels equally.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def api_preprocess_numpy(path_value):
    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()
    path = path_value.decode() if isinstance(path_value, bytes) else str(path_value)
    with Image.open(path) as image:
        image = image.convert("RGB")
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.NEAREST)
        return np.asarray(image, dtype=np.float32) / 255.0

def prepare_example(path, label, record_id):
    image = tf.numpy_function(api_preprocess_numpy, [path], tf.float32)
    image = tf.ensure_shape(image, [IMG_SIZE, IMG_SIZE, 3])
    return image, tf.reshape(tf.cast(label, tf.float32), [1]), record_id

def rows_to_dataset(rows, training=False, include_ids=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        rows["path"].astype(str).to_numpy(),
        rows["label"].to_numpy(np.int32),
        rows["record_id"].astype(str).to_numpy(),
    ))
    if training:
        dataset = dataset.shuffle(max(512, len(rows)), SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(prepare_example, num_parallel_calls=AUTOTUNE)
    if not include_ids:
        dataset = dataset.map(lambda image, label, _: (image, label), num_parallel_calls=AUTOTUNE)
    return dataset

def balanced_training_dataset(rows):
    positive = rows_to_dataset(rows[rows["label"] == 1], training=True).repeat()
    negative = rows_to_dataset(rows[rows["label"] == 0], training=True).repeat()
    dataset = tf.data.Dataset.sample_from_datasets(
        [positive, negative], weights=[0.5, 0.5], seed=SEED
    )
    return dataset.batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE)

train_rows = manifest[manifest["partition"] == "train"].copy()
validation_rows = manifest[manifest["partition"] == "validation"].copy()
calibration_rows = manifest[manifest["partition"] == "calibration"].copy()
test_rows = manifest[manifest["partition"] == "test"].copy()
mining_rows = manifest[manifest["partition"] == "mining"].copy()

train_ds = balanced_training_dataset(train_rows)
validation_ds = rows_to_dataset(validation_rows).batch(BATCH_SIZE).prefetch(AUTOTUNE)
calibration_ds = rows_to_dataset(calibration_rows, include_ids=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = rows_to_dataset(test_rows, include_ids=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)
mining_ds = rows_to_dataset(mining_rows, include_ids=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)
STEPS_PER_EPOCH = max(1, 2 * int(train_rows["label"].value_counts().min()) // BATCH_SIZE)
print("Balanced steps per epoch:", STEPS_PER_EPOCH)


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.06, fill_mode="reflect", seed=SEED + 1),
    tf.keras.layers.RandomTranslation(0.08, 0.08, fill_mode="reflect", seed=SEED + 2),
    tf.keras.layers.RandomZoom((-0.15, 0.12), (-0.15, 0.12), fill_mode="reflect", seed=SEED + 3),
    tf.keras.layers.RandomContrast(0.20, seed=SEED + 4),
    tf.keras.layers.GaussianNoise(0.02, seed=SEED + 5),
], name="camera_augmentation")

def build_model():
    inputs = tf.keras.Input([IMG_SIZE, IMG_SIZE, 3], dtype=tf.float32, name="image")
    x = augmentation(inputs)
    x = tf.keras.layers.Rescaling(2.0, offset=-1.0, name="imagenet_range")(x)
    backbone = tf.keras.applications.EfficientNetV2S(
        include_top=False, include_preprocessing=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False
    x = backbone(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.40, seed=SEED)(x)
    x = tf.keras.layers.Dense(256, activation="gelu")(x)
    x = tf.keras.layers.Dropout(0.25, seed=SEED + 1)(x)
    outputs = tf.keras.layers.Dense(
        1, activation="sigmoid", dtype="float32", name="food_probability"
    )(x)
    return tf.keras.Model(inputs, outputs, name="food_not_food_v2"), backbone

def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate, weight_decay=1e-5, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.02),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="food_recall"),
        ],
    )

model, backbone = build_model()
compile_model(model, 3e-4)
model.summary()


## Transfer learning with resumable Drive checkpoints

In [ ]:
def callbacks(stage_name, patience):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            RUN_CHECKPOINT_DIR / f"{stage_name}.keras", monitor="val_loss",
            save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=max(2, patience // 2),
            min_lr=1e-7, verbose=1,
        ),
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=RUN_CHECKPOINT_DIR / f"{stage_name}_backup", delete_checkpoint=True,
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]

stage1_path = RUN_CHECKPOINT_DIR / "stage1_head.keras"
stage2_path = RUN_CHECKPOINT_DIR / "stage2_fine_tuned.keras"
stage1_done = RUN_CHECKPOINT_DIR / "stage1.complete"
stage2_done = RUN_CHECKPOINT_DIR / "stage2.complete"

def get_backbone(current_model):
    return next(
        layer for layer in current_model.layers
        if isinstance(layer, tf.keras.Model) and layer.name.startswith("efficientnetv2")
    )

if RESUME_FROM_DRIVE and stage2_path.exists() and stage2_done.exists():
    model = tf.keras.models.load_model(stage2_path, compile=False)
    backbone = get_backbone(model)
    print("Resumed stage 2")
else:
    if RESUME_FROM_DRIVE and stage1_path.exists() and stage1_done.exists():
        model = tf.keras.models.load_model(stage1_path, compile=False)
        backbone = get_backbone(model)
    else:
        model.fit(
            train_ds, validation_data=validation_ds, steps_per_epoch=STEPS_PER_EPOCH,
            epochs=HEAD_EPOCHS, callbacks=callbacks("stage1_head", 3),
        )
        stage1_done.write_text("complete")
    backbone.trainable = True
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, 2e-5)
    model.fit(
        train_ds, validation_data=validation_ds, steps_per_epoch=STEPS_PER_EPOCH,
        epochs=FINE_TUNE_EPOCHS, callbacks=callbacks("stage2_fine_tuned", 6),
    )
    stage2_done.write_text("complete")


## Trusted hard-example mining

The mining split has curated binary labels. Stage 3 adds the highest-scoring not-food examples and lowest-scoring food examples without touching validation, calibration, or test data.

In [ ]:
def collect_predictions(current_model, dataset, description):
    labels, scores, record_ids = [], [], []
    for images, batch_labels, batch_ids in tqdm(dataset, desc=description):
        batch_scores = current_model(images, training=False).numpy().reshape(-1)
        labels.extend(batch_labels.numpy().reshape(-1).astype(np.int32))
        scores.extend(batch_scores)
        record_ids.extend(value.decode() for value in batch_ids.numpy())
    return np.asarray(labels), np.asarray(scores), record_ids

model = tf.keras.models.load_model(stage2_path, compile=False)
mining_labels, mining_scores, mining_ids = collect_predictions(model, mining_ds, "Mining")
mining_result = pd.DataFrame({
    "record_id": mining_ids, "label": mining_labels, "score": mining_scores,
})
hard_negative_ids = set(
    mining_result[mining_result["label"] == 0]
    .nlargest(MAX_HARD_PER_CLASS, "score")["record_id"]
)
hard_positive_ids = set(
    mining_result[mining_result["label"] == 1]
    .nsmallest(MAX_HARD_PER_CLASS, "score")["record_id"]
)
hard_ids = hard_negative_ids | hard_positive_ids
hard_rows = mining_rows[mining_rows["record_id"].isin(hard_ids)].copy()
hard_rows.to_csv(MANIFEST_DIR / f"hard_examples_{RUN_SIGNATURE}.csv", index=False)

hard_signature = hashlib.sha256("|".join(sorted(hard_ids)).encode()).hexdigest()[:12]
stage3_name = f"stage3_hard_{hard_signature}"
stage3_path = RUN_CHECKPOINT_DIR / f"{stage3_name}.keras"
stage3_done = RUN_CHECKPOINT_DIR / f"{stage3_name}.complete"
if RESUME_FROM_DRIVE and stage3_path.exists() and stage3_done.exists():
    model = tf.keras.models.load_model(stage3_path, compile=False)
else:
    train_with_hard = pd.concat([train_rows, hard_rows], ignore_index=True)
    train_with_hard_ds = balanced_training_dataset(train_with_hard)
    hard_steps = max(1, 2 * int(train_with_hard["label"].value_counts().min()) // BATCH_SIZE)
    compile_model(model, 5e-6)
    model.fit(
        train_with_hard_ds, validation_data=validation_ds, steps_per_epoch=hard_steps,
        epochs=HARD_EXAMPLE_EPOCHS, callbacks=callbacks(stage3_name, 3),
    )
    stage3_done.write_text("complete")


In [ ]:
checkpoint_paths = [
    path for path, marker in [
        (stage1_path, stage1_done), (stage2_path, stage2_done), (stage3_path, stage3_done),
    ] if path.exists() and marker.exists()
]
checkpoint_scores = []
validation_with_ids = rows_to_dataset(validation_rows, include_ids=True).batch(BATCH_SIZE)
for checkpoint_path in checkpoint_paths:
    candidate = tf.keras.models.load_model(checkpoint_path, compile=False)
    labels, scores, _ = collect_predictions(candidate, validation_with_ids, checkpoint_path.stem)
    scores = np.clip(scores, 1e-7, 1 - 1e-7)
    log_loss = float(-np.mean(labels * np.log(scores) + (1 - labels) * np.log(1 - scores)))
    checkpoint_scores.append((log_loss, checkpoint_path))
    print(checkpoint_path.name, log_loss)
best_loss, best_checkpoint = min(checkpoint_scores, key=lambda item: item[0])
model = tf.keras.models.load_model(best_checkpoint, compile=False)
print("Selected:", best_checkpoint)


## Calibration threshold and untouched test evaluation

In [ ]:
def select_threshold(labels, scores):
    results = []
    for threshold in np.linspace(0.0, 1.0, 1001):
        predictions = (scores >= threshold).astype(np.int32)
        tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
        food_recall = tp / max(1, tp + fn)
        specificity = tn / max(1, tn + fp)
        results.append((threshold, food_recall, specificity, (food_recall + specificity) / 2))
    table = pd.DataFrame(results, columns=["threshold", "food_recall", "specificity", "balanced_accuracy"])
    eligible = table[table["food_recall"] >= MIN_FOOD_RECALL]
    if eligible.empty:
        raise RuntimeError("No threshold satisfies the minimum food recall constraint")
    best = eligible.sort_values(["balanced_accuracy", "specificity"], ascending=False).iloc[0]
    return float(best["threshold"]), table

def metrics_at(labels, scores, threshold):
    predictions = (scores >= threshold).astype(np.int32)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "food_recall": float(recall_score(labels, predictions, zero_division=0)),
        "specificity": float(tn / max(1, tn + fp)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "pr_auc": float(average_precision_score(labels, scores)),
        "brier_score": float(brier_score_loss(labels, scores)),
        "confusion_matrix": matrix.tolist(),
    }

calibration_labels, calibration_scores, _ = collect_predictions(model, calibration_ds, "Calibration")
selected_threshold, threshold_table = select_threshold(calibration_labels, calibration_scores)
calibration_metrics = metrics_at(calibration_labels, calibration_scores, selected_threshold)
assert calibration_metrics["food_recall"] >= MIN_FOOD_RECALL

test_labels, test_scores, test_ids = collect_predictions(model, test_ds, "Test")
test_metrics = metrics_at(test_labels, test_scores, selected_threshold)
print("Threshold:", selected_threshold)
print(json.dumps(test_metrics, indent=2))
print(classification_report(
    test_labels, (test_scores >= selected_threshold).astype(np.int32),
    target_names=["not_food", "food"],
))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(
    np.asarray(test_metrics["confusion_matrix"]), annot=True, fmt="d", cmap="Blues",
    xticklabels=["not_food", "food"], yticklabels=["not_food", "food"], ax=axes[0],
)
fpr, tpr, _ = roc_curve(test_labels, test_scores)
axes[1].plot(fpr, tpr, label=f"AUC={test_metrics['roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].legend(); axes[1].set_title("ROC")
precision, recall, _ = precision_recall_curve(test_labels, test_scores)
axes[2].plot(recall, precision, label=f"AP={test_metrics['pr_auc']:.4f}")
axes[2].legend(); axes[2].set_title("Precision-recall")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / f"evaluation_{RUN_SIGNATURE}.png", dpi=180)
plt.show()


## Export, parity checks, CPU latency, and quality gate

In [ ]:
LOCAL_MODEL = LOCAL_ROOT / "food_or_not_food_detector_v2.keras"
FINAL_MODEL = ARTIFACT_DIR / LOCAL_MODEL.name
model.save(LOCAL_MODEL)
reloaded = tf.keras.models.load_model(LOCAL_MODEL, compile=False)
assert reloaded.input_shape == (None, IMG_SIZE, IMG_SIZE, 3)
assert reloaded.output_shape == (None, 1)

sample = np.random.default_rng(SEED).random((4, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
np.testing.assert_allclose(
    model(sample, training=False).numpy(), reloaded(sample, training=False).numpy(),
    rtol=1e-5, atol=1e-6,
)
api_sample = api_preprocess_numpy(test_rows.iloc[0]["path"])[None, ...]
api_score = float(reloaded(api_sample, training=False).numpy()[0, 0])
assert 0 <= api_score <= 1

with tf.device("/CPU:0"):
    cpu_model = tf.keras.models.load_model(LOCAL_MODEL, compile=False)
    cpu_sample = tf.random.uniform([1, IMG_SIZE, IMG_SIZE, 3])
    for _ in range(10): cpu_model(cpu_sample, training=False).numpy()
    start = time.perf_counter()
    for _ in range(50): cpu_model(cpu_sample, training=False).numpy()
    cpu_latency_ms = (time.perf_counter() - start) * 1000 / 50

quality_gate = {
    "balanced_accuracy": test_metrics["balanced_accuracy"] >= TARGET_BALANCED_ACCURACY,
    "food_recall": test_metrics["food_recall"] >= MIN_FOOD_RECALL,
    "specificity": test_metrics["specificity"] >= TARGET_SPECIFICITY,
    "cpu_latency": cpu_latency_ms <= CPU_SPEED_FLOOR_MS,
}
model_sha256 = hashlib.sha256(LOCAL_MODEL.read_bytes()).hexdigest()
threshold_artifact = {
    "threshold": selected_threshold, "positive_label": "food",
    "negative_label": "not_food", "input_size": [224, 224, 3],
    "input_range": [0.0, 1.0], "model_sha256": model_sha256,
    "minimum_food_recall_constraint": MIN_FOOD_RECALL,
}
metrics_artifact = {
    "calibration": calibration_metrics, "test": test_metrics,
    "cpu_latency_ms_colab": cpu_latency_ms, "quality_gate": quality_gate,
    "production_ready": all(quality_gate.values()),
    "dataset_revision": DATASET_REVISION, "run_signature": RUN_SIGNATURE,
}

shutil.copy2(LOCAL_MODEL, FINAL_MODEL)
(ARTIFACT_DIR / "food_not_food_threshold.json").write_text(json.dumps(threshold_artifact, indent=2))
(ARTIFACT_DIR / "food_not_food_metrics.json").write_text(json.dumps(metrics_artifact, indent=2))
print("Saved:", FINAL_MODEL)
print(f"CPU latency: {cpu_latency_ms:.1f} ms/image")
print("Quality gate:", quality_gate)
print("Production ready:", all(quality_gate.values()))


## Deployment

Download the `.keras`, threshold JSON, and metrics JSON from `MyDrive/NutrifyFoodDetectorV2LowDisk/artifacts`. Keep the old production detector for rollback, update the model path and threshold together in `main.py`, and test with real camera images including raw foods, prepared meals, empty plates, packaging, people, pets, screens, and low-light blur.

The compact benchmark covers 40 food and 40 not-food ImageNet classes. Add your own phone-camera examples in a future training iteration for stronger production accuracy.